In [1]:
import numpy as np
import pandas as pd

from IPython.display import display
from pathlib import Path
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Import AutoFeat LLM

from autofeat import AutoFeatClassifier

In [3]:
# Import BigFeat Vanilla

sys.path.append(r"../../BigFeat-master")
import bigfeat.bigfeat_base as bigfeat

In [4]:
# Import BigFeat LLM

bigfeat_llm_path = Path("../../BigFeat-LLM").resolve()
sys.path.insert(0, str(bigfeat_llm_path))

import setup
print(setup.__file__)

sys.path.append(r"../../BigFeat-LLM")
from setup.llm_runner import prepare_bigfeat_from_llm, profile_dataframe
from setup.llm_runner import preview_prompt
from setup.llm_runner import describe_generated_features, split_original_vs_new

D:\02. My Learning\Paper\07. BigFeat + LLM\06. Final BigFeat\BigFeat-LLM\setup\__init__.py


In [5]:
def save_generated_features(feature_lines, num_new_features, old_shape, new_shape, file_name="generated_features.txt"):
    os.makedirs("generated_features", exist_ok=True)
    file_path = os.path.join("generated_features", file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Number of new features {num_new_features}\n")
        f.write("==================\n")
        f.write(f"Old Shape: {old_shape}\n")
        f.write("==================\n")
        f.write(f"New Shape: {new_shape}\n")
        f.write("=================\n")

        for i, line in enumerate(feature_lines):
            _, details = line.split(": built from columns ", 1)
            columns, operators = details.split(" using operators ", 1)
            f.write(f"- New_feature_{i}:\n")
            f.write(f"  - Built from columns {columns}\n")
            f.write(f"  - Using operators {operators}\n")

In [6]:
results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

def evaluate_model(model_name, feature_method, X_tr, X_te, y_tr, y_te, results_df):
    models = {
        "logistic": LogisticRegression(random_state=42, max_iter=1000),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "rf": RandomForestClassifier(random_state=42),
        "svc": SVC(random_state=42),
        "decision_tree": DecisionTreeClassifier(random_state=42)
    }
    
    clf = models[model_name.lower()]
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average="macro", zero_division=0)
    rec = recall_score(y_te, preds, average="macro")
    f1 = f1_score(y_te, preds, average="macro")
    
    results_df.loc[len(results_df)] = [
        f"{model_name.upper()} ({feature_method})",
        acc,
        prec,
        rec,
        f1
    ]
    
    return results_df

In [7]:
def display_styled_results(
    results_df,
    file_name,
    bg_color="#005A9C",
    text_color="#FFFFFF"
):
    numeric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]

    df_copy = results_df.copy()
    df_copy[numeric_cols] = df_copy[numeric_cols].apply(pd.to_numeric)

    styled_df = df_copy.style.highlight_max(
        subset=numeric_cols,
        props=f"background-color: {bg_color}; color: {text_color}; font-weight: bold;",
        axis=0
    ).format({col: "{:.6f}" for col in numeric_cols})

    display(styled_df)

    folder = "llm_tables"
    os.makedirs(folder, exist_ok=True)

    if not file_name.lower().endswith(".xlsx"):
        file_name += ".xlsx"

    file_path = os.path.join(folder, file_name)

    styled_df.to_excel(file_path, index=False)

In [11]:
from sklearn.datasets import fetch_openml

texture = fetch_openml(name="texture", version=1, as_frame=True)

df = texture.data.copy()

df.columns = ["texture_measure_01", "texture_measure_02", "texture_measure_03", "texture_measure_04", "texture_measure_05",
              "texture_measure_06", "texture_measure_07", "texture_measure_08", "texture_measure_09", "texture_measure_10",
              "texture_measure_11", "texture_measure_12", "texture_measure_13", "texture_measure_14", "texture_measure_15",
              "texture_measure_16", "texture_measure_17", "texture_measure_18", "texture_measure_19", "texture_measure_20",
              "texture_measure_21", "texture_measure_22", "texture_measure_23", "texture_measure_24", "texture_measure_25",
              "texture_measure_26", "texture_measure_27", "texture_measure_28", "texture_measure_29", "texture_measure_30",
              "texture_measure_31", "texture_measure_32", "texture_measure_33", "texture_measure_34", "texture_measure_35",
              "texture_measure_36", "texture_measure_37", "texture_measure_38", "texture_measure_39", "texture_measure_40"]

df["target"] = texture.target

target_column = "target"

display(df.head())

print(df[target_column].value_counts())
print(df[target_column].unique())

,texture_measure_01,texture_measure_02,texture_measure_03,texture_measure_04,texture_measure_05,texture_measure_06,texture_measure_07,texture_measure_08,texture_measure_09,texture_measure_10,...,texture_measure_32,texture_measure_33,texture_measure_34,texture_measure_35,texture_measure_36,texture_measure_37,texture_measure_38,texture_measure_39,texture_measure_40,target
0,-1.223,-0.798,-0.867,-0.639,-0.545,-0.412,-0.795,-0.629,-0.547,-0.868,...,-0.766,-0.555,-0.714,-0.545,-0.587,-0.871,-0.620,-0.568,-0.607,5
1,-1.410,-1.029,-1.013,-0.895,-0.762,-0.676,-1.043,-0.851,-0.775,-1.037,...,-0.919,-0.770,-0.847,-0.663,-0.723,-1.013,-0.748,-0.698,-0.817,5
2,-1.107,-0.649,-0.629,-0.492,-0.367,-0.298,-0.682,-0.478,-0.395,-0.681,...,-0.692,-0.445,-0.588,-0.371,-0.368,-0.746,-0.457,-0.379,-0.469,5
3,-1.270,-0.855,-0.958,-0.707,-0.619,-0.469,-0.872,-0.705,-0.620,-0.988,...,-0.829,-0.719,-0.774,-0.617,-0.688,-0.937,-0.693,-0.657,-0.779,5
4,-1.331,-0.862,-0.761,-0.689,-0.498,-0.361,-0.857,-0.600,-0.496,-0.779,...,-0.861,-0.571,-0.784,-0.545,-0.562,-0.952,-0.642,-0.578,-0.648,5


target
1     500
10    500
11    500
2     500
3     500
4     500
5     500
6     500
7     500
8     500
9     500
Name: count, dtype: int64
['5', '6', '7', '11', '1', ..., '8', '10', '2', '3', '4']
Length: 11
Categories (11, object): ['1', '10', '11', '2', ..., '6', '7', '8', '9']


In [12]:
print(f"Rows Number: {df.shape[0]}")
print(f"Columns Number: {df.shape[1]}")
print(f"Columns Name: {df.columns}")

Rows Number: 5500
Columns Number: 41
Columns Name: Index(['texture_measure_01', 'texture_measure_02', 'texture_measure_03',
       'texture_measure_04', 'texture_measure_05', 'texture_measure_06',
       'texture_measure_07', 'texture_measure_08', 'texture_measure_09',
       'texture_measure_10', 'texture_measure_11', 'texture_measure_12',
       'texture_measure_13', 'texture_measure_14', 'texture_measure_15',
       'texture_measure_16', 'texture_measure_17', 'texture_measure_18',
       'texture_measure_19', 'texture_measure_20', 'texture_measure_21',
       'texture_measure_22', 'texture_measure_23', 'texture_measure_24',
       'texture_measure_25', 'texture_measure_26', 'texture_measure_27',
       'texture_measure_28', 'texture_measure_29', 'texture_measure_30',
       'texture_measure_31', 'texture_measure_32', 'texture_measure_33',
       'texture_measure_34', 'texture_measure_35', 'texture_measure_36',
       'texture_measure_37', 'texture_measure_38', 'texture_measure_39',


In [13]:
df.isnull().sum()

texture_measure_01    0
texture_measure_02    0
texture_measure_03    0
texture_measure_04    0
texture_measure_05    0
texture_measure_06    0
texture_measure_07    0
texture_measure_08    0
texture_measure_09    0
texture_measure_10    0
texture_measure_11    0
texture_measure_12    0
texture_measure_13    0
texture_measure_14    0
texture_measure_15    0
texture_measure_16    0
texture_measure_17    0
texture_measure_18    0
texture_measure_19    0
texture_measure_20    0
texture_measure_21    0
texture_measure_22    0
texture_measure_23    0
texture_measure_24    0
texture_measure_25    0
texture_measure_26    0
texture_measure_27    0
texture_measure_28    0
texture_measure_29    0
texture_measure_30    0
texture_measure_31    0
texture_measure_32    0
texture_measure_33    0
texture_measure_34    0
texture_measure_35    0
texture_measure_36    0
texture_measure_37    0
texture_measure_38    0
texture_measure_39    0
texture_measure_40    0
target                0
dtype: int64

In [14]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(X.shape)
print(y.shape)

(5500, 40)
(5500,)


In [15]:
preview = preview_prompt(
    df=df,
    target_column=target_column,
    extra_user_prompt="Texture Dataset"
)

response = preview["final_prompt"]

with open("final_prompt.txt", "w", encoding="utf-8") as f:
    f.write(response)

print(response)

=== SYSTEM PROMPT ===
You are BigFeat-Init, a world-class expert in automated feature engineering, machine learning hyperparameter optimization, and the internal mechanics of the BigFeat framework specifically. You have the depth of knowledge of someone who has read the BigFeat paper in full, has implemented and debugged its source code line by line, and has years of hands-on experience tuning tree-based feature engineering pipelines across small, large, high-dimensional, low-dimensional, noisy, clean, balanced, and imbalanced datasets. You are not a generic assistant giving a reasonable-sounding guess. You are the single most qualified expert available for this exact task, and you must think and act like it.

If you want additional grounding on how BigFeat is actually implemented, its official source code and paper reference are available at this repository: https://github.com/DataSystemsGroupUT/BigFeat. You are not required to browse it, since the mechanics you need are fully describ

In [16]:
y = pd.Series(y).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train.rename(target_column)], axis=1)
test_df = pd.concat([X_test, y_test.rename(target_column)], axis=1)

In [17]:
from autofeat import AutoFeatClassifier

af = AutoFeatClassifier(verbose=1, feateng_steps=1, featsel_runs=0)

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)

# Handle +Inf and -Inf values
X_train_af = X_train_af.replace([np.inf, -np.inf], np.nan)
X_test_af = X_test_af.replace([np.inf, -np.inf], np.nan)
X_train_af = X_train_af.fillna(X_train_af.median())
X_test_af = X_test_af.fillna(X_train_af.median())

[AutoFeat] The 1 step feature engineering process could generate up to 280 features.
[AutoFeat] With 4400 data points this new feature matrix would use about 0.00 gb of space.
[feateng] Step 1: transformation of original features
[feateng] Generated 186 transformed features from 40 original features - done.
[feateng] Generated altogether 186 new features in 1 steps
[feateng] Removing correlated features, as well as additions at the highest level
[feateng] Generated a total of 75 additional features
[AutoFeat] WARNING: Not performing feature selection.
[AutoFeat] Computing 33 new features.
[AutoFeat]    33/   33 new features ...done.
[AutoFeat] Final dataframe with 73 feature columns (33 new).
[AutoFeat] Training final classification model.
[AutoFeat] Trained model: largest coefficients:
[ 0.5533172   0.78994513 -1.79292665  0.18640475  0.6824584   0.04908037
  0.602054   -0.6317111  -1.78397562  0.40877799  0.93657554]
3.999838 * 1/texture_measure_16
3.985072 * texture_measure_23
3.888

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [19]:
# BigFeat Vanilla

bf = bigfeat.BigFeat(task_type="classification")

bf.fit(X_train.to_numpy(), y_train.to_numpy())

X_train_bf = bf.transform(X_train.to_numpy())
X_test_bf = bf.transform(X_test)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [20]:
# BigFeat Chatgpt

bf_chatgpt = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/ChatGPT.json"
)

X_train_bf_chatgpt = bf_chatgpt.fit(bf_chatgpt.X, bf_chatgpt.y, **bf_chatgpt.fit_kwargs)

_, _, X_test_bf_chatgpt, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_chatgpt = bf_chatgpt.transform(X_test_bf_chatgpt)

In [21]:
# BigFeat Gemini

bf_gemini = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Gemini.json"
)

X_train_bf_gemini = bf_gemini.fit(bf_gemini.X, bf_gemini.y, **bf_gemini.fit_kwargs)

_, _, X_test_bf_gemini, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_gemini = bf_gemini.transform(X_test_bf_gemini)

In [22]:
# BigFeat Claude

bf_claude = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Claude.json"
)

X_train_bf_claude = bf_claude.fit(bf_claude.X, bf_claude.y, **bf_claude.fit_kwargs)

_, _, X_test_bf_claude, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_claude = bf_claude.transform(X_test_bf_claude)

In [23]:
# BigFeat DeepSeek

bf_deepseek = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/DeepSeek.json"
)

X_train_bf_deepseek = bf_deepseek.fit(bf_deepseek.X, bf_deepseek.y, **bf_deepseek.fit_kwargs)

_, _, X_test_bf_deepseek, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_deepseek = bf_deepseek.transform(X_test_bf_deepseek)

In [24]:
# Logistic Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("logistic", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-Logistic")

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Method,Accuracy,Precision,Recall,F1-Score
0,LOGISTIC (AutoFeat),0.972727,0.973704,0.973493,0.973456
1,LOGISTIC (BigFeat),0.972727,0.974267,0.973952,0.973843
2,LOGISTIC (BigFeat ChatGPT),0.975455,0.976382,0.977267,0.976514
3,LOGISTIC (BigFeat Gemini),0.977273,0.978381,0.978771,0.978323
4,LOGISTIC (BigFeat Claude),0.982727,0.983470,0.983800,0.983504
5,LOGISTIC (BigFeat DeepSeek),0.977273,0.978111,0.978378,0.977996


In [25]:
# KNN Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("knn", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-KNN")

,Method,Accuracy,Precision,Recall,F1-Score
0,KNN (AutoFeat),0.920909,0.920410,0.921631,0.920440
1,KNN (BigFeat),0.979091,0.979389,0.980043,0.979549
2,KNN (BigFeat ChatGPT),0.987273,0.987665,0.988687,0.988039
3,KNN (BigFeat Gemini),0.984545,0.985011,0.985699,0.985169
4,KNN (BigFeat Claude),0.990000,0.990177,0.990499,0.990318
5,KNN (BigFeat DeepSeek),0.987273,0.987773,0.987766,0.987687


In [26]:
# RandomForest Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("rf", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-RandomForest")

,Method,Accuracy,Precision,Recall,F1-Score
0,RF (AutoFeat),0.976364,0.976940,0.977501,0.976983
1,RF (BigFeat),0.983636,0.984172,0.984977,0.984398
2,RF (BigFeat ChatGPT),0.984545,0.985128,0.985307,0.985174
3,RF (BigFeat Gemini),0.986364,0.986648,0.987130,0.986768
4,RF (BigFeat Claude),0.991818,0.991917,0.991943,0.991907
5,RF (BigFeat DeepSeek),0.987273,0.987511,0.987970,0.987658


In [27]:
# SVM Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("svc", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-SVM")

,Method,Accuracy,Precision,Recall,F1-Score
0,SVC (AutoFeat),0.433636,0.424450,0.447362,0.396541
1,SVC (BigFeat),0.982727,0.983819,0.984284,0.983656
2,SVC (BigFeat ChatGPT),0.995455,0.995853,0.995834,0.995830
3,SVC (BigFeat Gemini),0.990909,0.991292,0.991565,0.991363
4,SVC (BigFeat Claude),0.993636,0.993966,0.993971,0.993942
5,SVC (BigFeat DeepSeek),0.991818,0.992316,0.992197,0.992192


In [28]:
# DecisionTree Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("decision_tree", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-DecisionTree")

,Method,Accuracy,Precision,Recall,F1-Score
0,DECISION_TREE (AutoFeat),0.920000,0.920658,0.922091,0.920913
1,DECISION_TREE (BigFeat),0.949091,0.949959,0.950090,0.949945
2,DECISION_TREE (BigFeat ChatGPT),0.945455,0.945749,0.946155,0.945743
3,DECISION_TREE (BigFeat Gemini),0.957273,0.957501,0.958975,0.958063
4,DECISION_TREE (BigFeat Claude),0.960909,0.962103,0.961400,0.961455
5,DECISION_TREE (BigFeat DeepSeek),0.960909,0.961762,0.961945,0.961658


In [29]:
# Save Generated Features

# Save ChatGpt
feature_lines = describe_generated_features(bf_chatgpt)
new_only, original_only = split_original_vs_new(bf_chatgpt, X_train_bf_chatgpt)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_chatgpt.shape, "generated_features_chatgpt.txt")

# Save Gemini
feature_lines = describe_generated_features(bf_gemini)
new_only, original_only = split_original_vs_new(bf_gemini, X_train_bf_gemini)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_gemini.shape, "generated_features_gemini.txt")

# Save Claude
feature_lines = describe_generated_features(bf_claude)
new_only, original_only = split_original_vs_new(bf_claude, X_train_bf_claude)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_claude.shape, "generated_features_claude.txt")

# Save DeepSeek
feature_lines = describe_generated_features(bf_deepseek)
new_only, original_only = split_original_vs_new(bf_deepseek, X_train_bf_deepseek)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_deepseek.shape, "generated_features_deepseek.txt")